In [2]:
import os
import json

import torch
import torch.nn as nn
from umap import UMAP
from hdbscan import HDBSCAN
from sentence_transformers import SentenceTransformer
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from mpstemmer import MPStemmer
from transformers import BertTokenizer, BertModel
from bertopic import BERTopic
from bertopic.representation import MaximalMarginalRelevance
from bertopic.representation import OpenAI

import numpy as np
import pandas as pd

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print("Total GPU:", torch.cuda.device_count())
print("Current GPU:", torch.cuda.get_device_name(torch.cuda.current_device()))

Total GPU: 1
Current GPU: NVIDIA RTX A5000


## Before (Original / Baseline)

In [3]:
src = torch.randint(100, (3, 100))
tgt = torch.tensor([[30396,  1241,  2957,  5263,  6844,   448,  1224,  1470,  9289,    34,
         10561, 30470,     2,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0],
        [30396,   176,  2714, 30463,   119,   176,  3314,   342,    34,   119,
         21500, 30470, 30463,     2,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0],
        [30396,  8397,  7005,  7167, 29753,   305, 17529,  6075,  1300,  9780,
            26,  5209,  2627, 30468,  4750, 29948, 30468,   485,   386,  1530,
          4965,  4239,    16, 30469, 19052,    16, 30470, 30398,  4236,  1482,
           232, 10419, 30470]])

src_batch, src_len = src.size()
tgt_batch, tgt_len = tgt.size()

tgt_pad_mask = tgt.data.eq(0)
print("Tgt mask shape:", tgt_pad_mask.shape)
print("Tgt mask:", "\n", tgt_pad_mask)
print()

tgt_pad_mask = tgt.data.eq(0).unsqueeze(1)
print("Tgt mask shape:", tgt_pad_mask.shape)
print("Tgt mask:", "\n", tgt_pad_mask)
print()

tgt_pad_mask = tgt.data.eq(0).unsqueeze(1).expand(tgt_batch, tgt_len, tgt_len)
print("Tgt mask shape:", tgt_pad_mask.shape)
print("Tgt mask:", "\n", tgt_pad_mask)
print()

Tgt mask shape: torch.Size([3, 33])
Tgt mask: 
 tensor([[False, False, False, False, False, False, False, False, False, False,
         False, False, False,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True],
        [False, False, False, False, False, False, False, False, False, False,
         False, False, False, False,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True],
        [False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
         False, False, False]])

Tgt mask shape: torch.Size([3, 1, 33])
Tgt mask: 
 tensor([[[False, False, False, False, False, False, False, False, False, False,
          False

In [4]:
def get_attn_subsequent_mask(size):
    """
    Get an attention mask to avoid using the subsequent info.

    Args:
        size: int

    Returns:
        (`LongTensor`):

        * subsequent_mask `[1 x size x size]`
    """
    attn_shape = (1, size, size)
    subsequent_mask = np.triu(np.ones(attn_shape), k=1).astype('uint8')  # Create a matrix with triangular 0 values
    subsequent_mask = torch.from_numpy(subsequent_mask)
    return subsequent_mask

In [5]:
MAX_SIZE = 5000
mask = get_attn_subsequent_mask(MAX_SIZE)

# Generate a mask to prevent attending to subsequent positions in the target sequence.
dec_mask = torch.gt(tgt_pad_mask + mask[:, :tgt_pad_mask.size(1), :tgt_pad_mask.size(1)], 0)
print("Decoder mask shape:", dec_mask.shape)
print("Decoder mask:", "\n", dec_mask)

Decoder mask shape: torch.Size([3, 33, 33])
Decoder mask: 
 tensor([[[False,  True,  True,  ...,  True,  True,  True],
         [False, False,  True,  ...,  True,  True,  True],
         [False, False, False,  ...,  True,  True,  True],
         ...,
         [False, False, False,  ...,  True,  True,  True],
         [False, False, False,  ...,  True,  True,  True],
         [False, False, False,  ...,  True,  True,  True]],

        [[False,  True,  True,  ...,  True,  True,  True],
         [False, False,  True,  ...,  True,  True,  True],
         [False, False, False,  ...,  True,  True,  True],
         ...,
         [False, False, False,  ...,  True,  True,  True],
         [False, False, False,  ...,  True,  True,  True],
         [False, False, False,  ...,  True,  True,  True]],

        [[False,  True,  True,  ...,  True,  True,  True],
         [False, False,  True,  ...,  True,  True,  True],
         [False, False, False,  ...,  True,  True,  True],
         ...,
         

## After (Topic Embedding)

### Scenario Default

In [6]:
src = torch.randint(100, (3, 100))
tgt = torch.tensor([[30396,  1241,  2957,  5263,  6844, 0, 0, 0]])

src_batch, src_len = src.size()
tgt_batch, tgt_len = tgt.size()

tgt_pad_mask = tgt.data.eq(0)
print("Tgt mask shape:", tgt_pad_mask.shape)
print("Tgt mask:", "\n", tgt_pad_mask)
print()

tgt_pad_mask = tgt.data.eq(0).unsqueeze(1)
print("Tgt mask shape:", tgt_pad_mask.shape)
print("Tgt mask:", "\n", tgt_pad_mask)
print()

tgt_pad_mask = tgt.data.eq(0).unsqueeze(1).expand(tgt_batch, tgt_len, tgt_len)
print("Tgt mask shape:", tgt_pad_mask.shape)
print("Tgt mask:", "\n", tgt_pad_mask)
print()

Tgt mask shape: torch.Size([1, 8])
Tgt mask: 
 tensor([[False, False, False, False, False,  True,  True,  True]])

Tgt mask shape: torch.Size([1, 1, 8])
Tgt mask: 
 tensor([[[False, False, False, False, False,  True,  True,  True]]])

Tgt mask shape: torch.Size([1, 8, 8])
Tgt mask: 
 tensor([[[False, False, False, False, False,  True,  True,  True],
         [False, False, False, False, False,  True,  True,  True],
         [False, False, False, False, False,  True,  True,  True],
         [False, False, False, False, False,  True,  True,  True],
         [False, False, False, False, False,  True,  True,  True],
         [False, False, False, False, False,  True,  True,  True],
         [False, False, False, False, False,  True,  True,  True],
         [False, False, False, False, False,  True,  True,  True]]])



In [7]:
def get_attn_subsequent_mask_with_topic(topic_len, tgt_len):
    attn_shape = (1, tgt_len, topic_len + tgt_len)
    subsequent_mask = np.triu(np.ones(attn_shape), k=topic_len+1).astype('uint8')
    subsequent_mask = torch.from_numpy(subsequent_mask)
    return subsequent_mask

In [8]:
tgt_pad_mask = tgt.data.eq(0).unsqueeze(1).expand(tgt_batch, tgt_len, tgt_len)

topic_len = 3
tgt_len = 8
MAX_BATCH = 3
MAX_SIZE = 20
mask = get_attn_subsequent_mask_with_topic(topic_len, tgt_len)
print("Mask:")
print(mask)

topic_pad_mask = torch.zeros(MAX_BATCH, MAX_SIZE, topic_len)

topic_mask_tgt = abs(np.triu(np.ones((MAX_BATCH, topic_len, MAX_SIZE)), k=0) * 
                np.tril(np.ones((MAX_BATCH, topic_len, MAX_SIZE)), k=0) - 1).astype('uint8')
topic_mask_tgt = torch.from_numpy(topic_mask_tgt)
print("Topic mask tgt:")
print(topic_mask_tgt)

topic_mask_src = torch.ones(MAX_BATCH, topic_len, MAX_SIZE)

# Mask for SA decoder
# Phase 1
tgt_pad_mask = torch.cat((tgt_pad_mask, topic_pad_mask[:tgt_pad_mask.size(0), :tgt_pad_mask.size(1), :]), axis=2)
print("Tgt pad mask:")
print(tgt_pad_mask)
tgt_pad_mask = tgt_pad_mask + mask[:, :tgt_pad_mask.size(1), :tgt_pad_mask.size(2)]
print("Tgt pad mask:")
print(tgt_pad_mask)

# Phase 2
topic_batch = tgt_pad_mask.size(0)
total_len = tgt_pad_mask.size(2)

# Phase 3
print("Topic mask tgt:")
print(topic_mask_tgt[:topic_batch, :topic_len, :total_len])
dec_mask = torch.cat((topic_mask_tgt[:topic_batch, :topic_len, :total_len], tgt_pad_mask), axis=1)
print("Dec mask:")
print(dec_mask)
dec_mask = torch.gt(dec_mask, 0)

Mask:
tensor([[[0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1],
         [0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1],
         [0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1],
         [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1],
         [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1],
         [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1],
         [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
         [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]], dtype=torch.uint8)
Topic mask tgt:
tensor([[[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
         [1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
         [1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]],

        [[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
         [1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
         [1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]],

        [[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
         [1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
         [1, 1, 0, 1, 1, 1, 1, 1,

### Scenario Concat Topic Embedding at the End

In [9]:
def get_attn_subsequent_mask_reversed(topic_len, tgt_len):
    attn_shape = (1, tgt_len, tgt_len)
    addition_shape = (1, tgt_len, topic_len)
    subsequent_mask = np.triu(np.ones(attn_shape), k=1)
    subsequent_mask = np.concatenate((subsequent_mask, np.zeros(addition_shape)), axis=2).astype('uint8')
    subsequent_mask = torch.from_numpy(subsequent_mask)
    return subsequent_mask

In [11]:
tgt_pad_mask = tgt.data.eq(0).unsqueeze(1).expand(tgt_batch, tgt_len, tgt_len)

topic_len = 3
tgt_len = 8
MAX_BATCH = 3
MAX_SIZE = 20
mask = get_attn_subsequent_mask_reversed(topic_len, tgt_len)
print("Mask:")
print(mask)

topic_pad_mask = torch.zeros(MAX_BATCH, MAX_SIZE, topic_len)
# print(topic_pad_mask)

topic_mask_tgt = abs(np.triu(np.ones((MAX_BATCH, topic_len, MAX_SIZE)), k=MAX_SIZE-topic_len) * 
                np.tril(np.ones((MAX_BATCH, topic_len, MAX_SIZE)), k=MAX_SIZE-topic_len) - 1).astype('uint8')
topic_mask_tgt = torch.from_numpy(topic_mask_tgt)
print("Topic mask tgt:")
print(topic_mask_tgt)

topic_mask_src = torch.ones(MAX_BATCH, topic_len, MAX_SIZE)
# print(topic_mask_src)

# Mask for SA decoder
# Phase 1
# print("Tgt pad mask:")
# print(tgt_pad_mask)
tgt_pad_mask = torch.cat((tgt_pad_mask, topic_pad_mask[:tgt_pad_mask.size(0), :tgt_pad_mask.size(1), :]), axis=2)
print("Tgt pad mask:")
# print(tgt_pad_mask)
tgt_pad_mask = tgt_pad_mask + mask[:, :tgt_pad_mask.size(1), :tgt_pad_mask.size(2)]
print(tgt_pad_mask)

# Phase 2
topic_batch = tgt_pad_mask.size(0)
total_len = tgt_pad_mask.size(2)

# # Phase 3
print("Topic mask tgt:")
print(topic_mask_tgt[:topic_batch, :topic_len, -total_len:])

dec_mask = torch.cat((tgt_pad_mask, topic_mask_tgt[:topic_batch, :topic_len, -total_len:]), axis=1)
print("Dec mask:")
print(dec_mask)
# dec_mask = torch.gt(dec_mask, 0)

Mask:
tensor([[[0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0],
         [0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0],
         [0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0],
         [0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0],
         [0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]], dtype=torch.uint8)
Topic mask tgt:
tensor([[[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1],
         [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1],
         [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0]],

        [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1],
         [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1],
         [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0]],

        [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1],
         [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1],
         [1, 1, 1, 1, 1, 1, 1, 1,

### Scenario Change Topic Mask TGT

In [12]:
# Mask Full 
tgt_pad_mask = tgt.data.eq(0).unsqueeze(1).expand(tgt_batch, tgt_len, tgt_len)

topic_len = 3
tgt_len = 8
MAX_BATCH = 3
MAX_SIZE = 20
mask = get_attn_subsequent_mask_with_topic(topic_len, tgt_len)
# print("Mask:")
# print(mask)

topic_pad_mask = torch.zeros(MAX_BATCH, MAX_SIZE, topic_len)
# print(topic_pad_mask)

topic_mask_tgt = np.zeros((MAX_BATCH, topic_len, topic_len))
topic_mask_tgt = np.concatenate((topic_mask_tgt, np.ones((MAX_BATCH, topic_len, MAX_SIZE-topic_len))), axis=2).astype('uint8')
topic_mask_tgt = torch.from_numpy(topic_mask_tgt)
print("Topic mask tgt:")
print(topic_mask_tgt)

topic_mask_src = torch.ones(MAX_BATCH, topic_len, MAX_SIZE)
# print(topic_mask_src)

# Mask for SA decoder
# Phase 1
tgt_pad_mask = torch.cat((tgt_pad_mask, topic_pad_mask[:tgt_pad_mask.size(0), :tgt_pad_mask.size(1), :]), axis=2)
# print("Tgt pad mask:")
# print(tgt_pad_mask)
tgt_pad_mask = tgt_pad_mask + mask[:, :tgt_pad_mask.size(1), :tgt_pad_mask.size(2)]
# print("Tgt pad mask:")
# print(tgt_pad_mask)

# Phase 2
topic_batch = tgt_pad_mask.size(0)
total_len = tgt_pad_mask.size(2)

# Phase 3
# print("Topic mask tgt:")
# print(topic_mask_tgt[:topic_batch, :topic_len, :total_len])
dec_mask = torch.cat((topic_mask_tgt[:topic_batch, :topic_len, :total_len], tgt_pad_mask), axis=1)
# print("Dec mask:")
# print(dec_mask)
dec_mask = torch.gt(dec_mask, 0)


Topic mask tgt:
tensor([[[0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
         [0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
         [0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]],

        [[0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
         [0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
         [0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]],

        [[0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
         [0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
         [0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]],
       dtype=torch.uint8)


In [13]:
# Mask similar with token emb 
tgt_pad_mask = tgt.data.eq(0).unsqueeze(1).expand(tgt_batch, tgt_len, tgt_len)

topic_len = 3
tgt_len = 8
MAX_BATCH = 3
MAX_SIZE = 20
mask = get_attn_subsequent_mask_with_topic(topic_len, tgt_len)
# print("Mask:")
# print(mask)

topic_pad_mask = torch.zeros(MAX_BATCH, MAX_SIZE, topic_len)
# print(topic_pad_mask)

# topic_mask_tgt = np.zeros((MAX_BATCH, topic_len, topic_len))
# topic_mask_tgt = np.concatenate((topic_mask_tgt, np.ones((MAX_BATCH, topic_len, MAX_SIZE-topic_len))), axis=2).astype('uint8')
topic_mask_tgt = np.triu(np.ones((MAX_BATCH, topic_len, MAX_SIZE)), k=1).astype("int8")
topic_mask_tgt = torch.from_numpy(topic_mask_tgt)
print("Topic mask tgt:")
print(topic_mask_tgt)

topic_mask_src = torch.ones(MAX_BATCH, topic_len, MAX_SIZE)
# print(topic_mask_src)

# Mask for SA decoder
# Phase 1
tgt_pad_mask = torch.cat((tgt_pad_mask, topic_pad_mask[:tgt_pad_mask.size(0), :tgt_pad_mask.size(1), :]), axis=2)
# print("Tgt pad mask:")
# print(tgt_pad_mask)
tgt_pad_mask = tgt_pad_mask + mask[:, :tgt_pad_mask.size(1), :tgt_pad_mask.size(2)]
# print("Tgt pad mask:")
# print(tgt_pad_mask)

# Phase 2
topic_batch = tgt_pad_mask.size(0)
total_len = tgt_pad_mask.size(2)

# Phase 3
# print("Topic mask tgt:")
# print(topic_mask_tgt[:topic_batch, :topic_len, :total_len])
dec_mask = torch.cat((topic_mask_tgt[:topic_batch, :topic_len, :total_len], tgt_pad_mask), axis=1)
# print("Dec mask:")
# print(dec_mask)
dec_mask = torch.gt(dec_mask, 0)


Topic mask tgt:
tensor([[[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
         [0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
         [0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]],

        [[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
         [0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
         [0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]],

        [[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
         [0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
         [0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]],
       dtype=torch.int8)
